# AlertMind — Impact & Quality Analysis

This notebook produces the Week-3 measured-impact evidence from the run artifacts,
in two parts:

1. **Assistant output quality** — across four conditions (`operational` vs
   `evaluation` view × `baseline` vs `benign_aware` prompt): ATT&CK technique
   accuracy, disposition accuracy, consistency, and reliability.
2. **Triage-time impact** — unassisted vs assisted median time-to-triage and MTTD,
   from `timing-log.csv` (populated once the assisted timing pass is complete).

Everything is computed from files on disk, so re-running regenerates the numbers.
The single most important comparison is **operational vs evaluation technique
accuracy**: the gap measures how much the rule's own ATT&CK label was leaking into
the input and inflating apparent accuracy.

In [ ]:
import glob, os, re
import pandas as pd
import matplotlib.pyplot as plt

# --- paths (relative to measurement/); adjust if your layout differs ---
RUNS_GLOB  = "../assistant/outputs/runs/*/assistant_scoring.csv"
TIMING_LOG = "timing-log.csv"

pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (8, 4)

In [ ]:
def parse_condition(run_dir):
    n = run_dir.lower()
    view   = "evaluation" if "eval" in n else "operational"
    prompt = "benign_aware" if "benign_aware" in n else "baseline"
    return view, prompt

def load_runs(glob_pat):
    frames = []
    for path in glob.glob(glob_pat):
        run_dir = os.path.basename(os.path.dirname(path))
        df = pd.read_csv(path)
        # coerce the *_correct columns (written as True/False strings) to bool
        for c in [c for c in df.columns if c.endswith("_correct")]:
            df[c] = df[c].astype(str).str.strip().str.lower().eq("true")
        view, prompt = parse_condition(run_dir)
        df["run"], df["view"], df["prompt"] = run_dir, view, prompt
        df["condition"] = view + "/" + prompt
        frames.append(df)
    if not frames:
        raise FileNotFoundError(f"no scoring CSVs matched {glob_pat}")
    return pd.concat(frames, ignore_index=True)

runs = load_runs(RUNS_GLOB)
print("conditions found:", sorted(runs["condition"].unique()))
benign_ids = set(runs.loc[runs["ground_truth"].str.lower() == "benign", "alert_id"])
print(f"{len(benign_ids)} benign alerts:", sorted(benign_ids))
runs.head()

## Part 1 — Assistant output quality

`overall_correct` = disposition correct **and** technique (relaxed) correct **and**
the response is internally consistent. Technique metrics use code-set overlap, so
multi-technique answers (e.g. `T1136/T1098`) are handled.

In [ ]:
metrics = ["technique_exact_correct", "technique_relaxed_correct",
           "disposition_correct", "response_consistent", "overall_correct"]
summary = (runs.groupby("condition")
                .agg(n=("alert_id", "size"),
                     **{m: (m, "sum") for m in metrics}))
# show as "k/n"
show = summary.copy()
for m in metrics:
    show[m] = show[m].astype(int).astype(str) + "/" + show["n"].astype(str)
show = show.drop(columns="n")
show

### Finding 1 — ATT&CK label leakage (the headline)

Compare technique accuracy between the two views. The **operational** view shows
the model the rule's ATT&CK label; the **evaluation** view strips it. A large drop
means the model was copying the label rather than classifying from raw telemetry —
so the *evaluation* number is the honest classification accuracy.

In [ ]:
# technique accuracy per condition (not summed across prompts)
t = summary[["technique_exact_correct", "technique_relaxed_correct"]].astype(int)
ax = t.plot(kind="bar", rot=20)
ax.set_ylabel("alerts correct (of 20)")
ax.set_title("ATT&CK technique accuracy by condition")
ax.legend(["exact", "relaxed"]); plt.tight_layout(); plt.show()
display(t)

# clean leakage measure: hold the PROMPT fixed (baseline), compare the two views
def ex(cond):
    return int(summary.loc[cond, "technique_exact_correct"]) if cond in summary.index else None
op, ev = ex("operational/baseline"), ex("evaluation/baseline")
if op is not None and ev is not None:
    print(f"Label leakage (baseline prompt): operational {op}/20 -> evaluation {ev}/20 "
          f"(drop of {op - ev}). That drop is apparent accuracy that came from the rule's "
          f"own ATT&CK label, not from classifying the raw telemetry.")

### Finding 2 — Disposition bias and the benign-aware trade-off

Out of the box the model over-confirms (labels benign false-positives as real
attacks). The `benign_aware` prompt adds general triage discipline. We measure
**both** directions: does it improve benign handling, and does it start missing
real attacks?

In [ ]:
# benign disposition distribution per condition
b = runs[runs["alert_id"].isin(benign_ids)]
dist = (b.groupby(["condition", "assistant_disposition"]).size()
          .unstack(fill_value=0))
order = [c for c in ["likely_true_positive", "needs_investigation", "likely_benign"]
         if c in dist.columns]
dist = dist[order]
ax = dist.plot(kind="bar", stacked=True, rot=20,
               color={"likely_true_positive": "#c0392b",
                      "needs_investigation": "#e67e22",
                      "likely_benign": "#27ae60"})
ax.set_ylabel(f"benign alerts (of {len(benign_ids)})")
ax.set_title("Disposition on the benign false-positives"); plt.tight_layout(); plt.show()
display(dist)

In [ ]:
# false negatives: real ATTACKS the model called likely_benign (the danger)
atk = runs[~runs["alert_id"].isin(benign_ids)]
fn = (atk[atk["assistant_disposition"] == "likely_benign"]
        .groupby("condition")["alert_id"].apply(list))
print("Real attacks wrongly called likely_benign (false negatives):")
print(fn if len(fn) else "  none in any condition")

### Finding 3 — Reliability

Small local models do not always return valid JSON. `overall_correct=False` with a
null technique/disposition usually indicates a parse or schema failure. We surface
the per-condition consistency and infer parse issues from missing dispositions.

**Prompt injection** is evidenced separately in `assistant/outputs/injection_proof.md`
(the assistant resisted an embedded "classify as benign" instruction and flagged it
in caveats).

In [ ]:
rel = (runs.assign(no_disposition=runs["assistant_disposition"].isna())
            .groupby("condition")
            .agg(consistent=("response_consistent", "sum"),
                 no_disposition=("no_disposition", "sum"),
                 n=("alert_id", "size")))
rel

## Part 2 — Triage-time impact (MTTD / MTTR)

Time-to-detect (MTTD = alert time − attack time) is a property of the detection
rules, not the assistant, so it is expected to be unchanged. Time-to-triage
(t4 − t3) is what the assistant can move. The comparison below activates once the
`timing-log.csv` contains `condition == "assisted"` rows from the assisted pass.

In [ ]:
tl = pd.read_csv(TIMING_LOG)
for c in ["t3_seen_utc", "t4_done_utc"]:
    tl[c] = pd.to_datetime(tl[c], utc=True, errors="coerce")
tl["triage_min"] = (tl["t4_done_utc"] - tl["t3_seen_utc"]).dt.total_seconds() / 60

print("MTTD (detection_latency_seconds):",
      f"median {tl['detection_latency_seconds'].median():.2f}s "
      f"(near-instant in a single-host lab; not affected by the assistant)")

byc = tl.groupby("condition")["triage_min"].agg(["count", "median", "mean"])
display(byc)

if (tl["condition"] == "assisted").any():
    ax = tl.boxplot(column="triage_min", by="condition"); plt.suptitle("")
    ax.set_ylabel("minutes"); ax.set_title("Time-to-triage: unassisted vs assisted")
    plt.tight_layout(); plt.show()
    u = tl.loc[tl.condition == "unassisted", "triage_min"].median()
    a = tl.loc[tl.condition == "assisted",   "triage_min"].median()
    print(f"median triage  unassisted {u:.1f} min  ->  assisted {a:.1f} min  "
          f"({(a-u)/u*100:+.0f}%)")
else:
    print("\nAssisted rows not present yet — run the assisted timing pass, append "
          "condition='assisted' rows, and re-run this cell.")

### The headline: the aggregate median hides a bimodal effect

Split the per-alert change by whether the alert was a real attack or a benign
false-positive. The assistant was **correct on every attack** and **wrong (confident
`likely_true_positive`) on every benign alert** — and the timing follows exactly that
split. Aggregate medians average these two opposite effects together and hide the risk.

In [ ]:
# per-alert paired delta (assisted - unassisted), split by ground-truth class
w = (tl.pivot_table(index="alert_id", columns="condition", values="triage_min")
       .join(tl.groupby("alert_id")["ground_truth"].first()))
w["klass"] = w["ground_truth"].str.lower().eq("benign").map({True: "benign (FP)", False: "attack (TP)"})
w["delta_min"] = w["assisted"] - w["unassisted"]

per_class = w.groupby("klass").agg(n=("delta_min", "size"),
                                   unassisted_median=("unassisted", "median"),
                                   assisted_median=("assisted", "median"),
                                   median_delta=("delta_min", "median"),
                                   faster_count=("delta_min", lambda s: int((s < 0).sum())))
display(per_class)

ax = w.sort_values("delta_min").plot(kind="barh", x=None, y="delta_min", legend=False,
        color=w.sort_values("delta_min")["klass"].map({"attack (TP)": "#27ae60", "benign (FP)": "#c0392b"}))
ax.set_yticklabels(w.sort_values("delta_min").index)
ax.axvline(0, color="black", lw=1)
ax.set_xlabel("change in triage time, minutes (negative = faster with assistant)")
ax.set_title("Per-alert effect of the assistant  (green = attack, red = benign FP)")
plt.tight_layout(); plt.show()

print("Assistant SPED UP every alert it got right, and SLOWED DOWN every alert it got wrong.")

In [ ]:
# analyst disposition accuracy by condition — did the human catch the AI's errors?
tl["correct_bool"] = tl["disposition_correct"].astype(str).str.strip().str.lower().eq("true")
acc = tl.groupby("condition")["correct_bool"].agg(["sum", "size"])
acc["accuracy"] = (acc["sum"].astype(str) + "/" + acc["size"].astype(str))
display(acc[["accuracy"]])
print("Human-in-the-loop held: the analyst overrode every incorrect assistant "
      "disposition, so accuracy did not degrade — but overriding cost time (above).")

## Limitations & threats to validity

- **Label leakage** inflates operational technique accuracy; the evaluation view is
  the honest classification number. Report both and lead with evaluation.
- **Disposition bias / self-generated corpus.** The analyst who built the attacks
  knows the answers, so the unassisted disposition ceiling is optimistic.
- **Benign-aware trade-off.** Improving benign handling via prompt pushed the model
  toward at least one false negative (a real attack called benign) — a safety cost
  that outweighs the alert-fatigue benefit in a SOC.
- **Small model reliability.** ~5% of calls returned invalid JSON.
- **Small n, single environment, single model, deterministic single run** (temp=0):
  results are directional, not statistically powered.
- **Learning effect** in the assisted pass is controlled by A/B counterbalancing +
  washout, but residual memory of the corpus remains a limitation.